# All Fastchem Gas and Cond

Hajime Kawahara 2025/11/27



In [1]:
from jax import config
config.update("jax_enable_x64", True)

We assume N2+H2O (gas, water, ice) system using fastchem/fastchem_cond presets. 

In [2]:
from exogibbs.presets.fastchem_cond import chemsetup as condsetup
cond = condsetup()
from exogibbs.presets.fastchem import chemsetup as gassetup
gas = gassetup()


fastchem_cond presets in ExoGibbs
number of species: 186 elements: 28 molecules: 186
fastchem presets in ExoGibbs
number of species: 523 elements: 28 molecules: 495


In [3]:
import numpy as np
from exojax.utils.zsol import nsol
import jax.numpy as jnp

solar_abundance = nsol()
nsol_vector = jnp.array(
    [solar_abundance[el] for el in gas.elements[:-1]]
)  # no solar abundance for e-
element_vector = jnp.append(nsol_vector, 0.0)

formula_matrix_gas = gas.formula_matrix

print("Formula matrix (gas):")
print(formula_matrix_gas)

formula_matrix_cond = cond.formula_matrix

print("Formula matrix (cond):")
print(formula_matrix_cond)

b_ref = gas.element_vector_reference

Database for solar abundance =  AAG21
Asplund, M., Amarsi, A. M., & Grevesse, N. 2021, arXiv:2105.01661
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


This setting yields rank(Ac, Ag) < |b_element| because formula_matrix_gas[:,0] = formula_matrix_cond[:,0]. We need to redefine the formulation using the matrix contraction. 

In [4]:
from exogibbs.thermo.stoichiometry import contract_formula_matrix
formula_matrix_gas_eff, formula_matrix_cond_eff, indep_element_mask = contract_formula_matrix(formula_matrix_gas, formula_matrix_cond)
#elements_eff =elements[indep_element_mask]

print("Formula matrix (gas):")
print(formula_matrix_gas_eff)
print("Formula matrix (cond):")
print(formula_matrix_cond_eff)
#print("independent elements:")
#print(elements_eff)


No contraction of the system needed.
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Output the reference-state value of ( $h = \mu / (RT)$ ) at temperature ( T ).


## minimization using minimize_gibbs_cond_core

In [5]:
from exogibbs.optimize.minimize_cond import minimize_gibbs_cond_core
import jax.numpy as jnp
from exogibbs.api.chemistry import ThermoState

from exogibbs.optimize.core import compute_ln_normalized_pressure


In [18]:

# Thermodynamic conditions
P = 1.0  # bar
Pref = 1.0  # bar, reference pressure
ln_normalized_pressure = compute_ln_normalized_pressure(P, Pref)

# Initial guess for log number densities
ln_etak = jnp.zeros(formula_matrix_cond_eff.shape[1])  # log(eta)
ln_nk = jnp.zeros(formula_matrix_gas_eff.shape[1])  # log(n_species)
ln_mk = jnp.zeros(formula_matrix_cond_eff.shape[1]) - 30.0  # log(n_condensates)
ln_ntot = 0.0  # log(total number density)

epsilon = -20.0
for i, temperature in enumerate([200.0]):

    thermo_state = ThermoState(
        temperature=temperature,
        ln_normalized_pressure=ln_normalized_pressure,
        element_vector=b_ref,
    )

    ln_nk, ln_mk, ln_etak, ln_ntot, counter = minimize_gibbs_cond_core(
        thermo_state,
        ln_nk_init=ln_nk,
        ln_mk_init=ln_mk,
        ln_etak_init=ln_etak,
        ln_ntot_init=ln_ntot,
        formula_matrix=formula_matrix_gas_eff,
        formula_matrix_cond=formula_matrix_cond_eff,
        hvector_func=gas.hvector_func,
        hvector_cond_func=cond.hvector_func,
        epsilon=epsilon,  ### new argument
        residual_crit=1.0e-11,
        max_iter=2,
    )


    print(formula_matrix_gas_eff@jnp.exp(ln_nk) + formula_matrix_cond_eff @ jnp.exp(ln_mk) - element_vector )
    print(jnp.max(jnp.exp(ln_etak)), jnp.max(jnp.exp(ln_nk)), jnp.max(jnp.exp(ln_mk)), jnp.max(jnp.exp(ln_ntot)), jnp.exp(epsilon))
    

[inf nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan inf nan nan inf nan nan nan nan]
201178.66136923764 212.03560618554184 inf 392.8675471912586 2.061153622438558e-09
